# 21｜不用预制 LSTM/CRF：手写 BiLSTM-CRF 序列标注

本笔记在单文件内重新实现 LSTM cell、双向变长扫描器、线性链 CRF 的 gold score、log-partition、NLL 与 Viterbi，再组合成 `BiLSTMCRF.forward`。我们用穷举验证动态规划，而不是仅凭训练 loss 猜实现正确。

## 1. 数据与状态合同

- token/tag/mask 形状都是 `[B,T]`，mask 必须是左对齐连续前缀且每条至少一个 token。
- emission 为 `[B,T,K]`；`transition[next_tag, previous_tag]`，不能混淆方向。
- START/END 不作为普通标签加入输出空间，而由独立参数表示。
- 约束采用布尔 `allowed_*`；非法 gold 路径立即拒绝。
- 标签采用 BIO：`0=O, 1=B-X, 2=I-X`。

In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

import hashlib  # 导入本单元所需的依赖。
import io  # 导入本单元所需的依赖。
import itertools  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。

import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 20260730  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。
O, B_X, I_X = 0, 1, 2  # 计算并保存当前步骤的中间状态。
TAG_NAMES = ["O", "B-X", "I-X"]  # 计算并保存当前步骤的中间状态。

assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
assert len(TAG_NAMES) == 3  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "tags": TAG_NAMES})  # 执行当前语句以推进本节示例。

## 2. 单步 LSTM 与双向扫描

LSTM 使用 `i,f,g,o` 四门：$c_t=f_t\odot c_{t-1}+i_t\odot g_t$，$h_t=o_t\odot\tanh(c_t)$。双向网络并不是调用 `bidirectional=True`：前向按 $0\to T-1$ 扫描，反向按 $T-1\to0$ 扫描，两边在无效位置均冻结状态并把输出清零，最后拼成 `[B,T,2H]`。

In [ ]:
class ScratchLSTMCell(nn.Module):  # 定义承载本节状态与行为的数据结构。
    gate_order = "ifgo"  # 计算并保存当前步骤的中间状态。
    def __init__(self, input_size, hidden_size):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.input_size, self.hidden_size = input_size, hidden_size  # 计算并保存当前步骤的中间状态。
        self.gates = nn.Linear(input_size + hidden_size, 4 * hidden_size)  # 计算并保存当前步骤的中间状态。
    def forward(self, x_t, state):  # 定义本节可复用的核心函数。
        h, c = state  # 计算并保存当前步骤的中间状态。
        i, f, g, o = self.gates(torch.cat([x_t, h], -1)).chunk(4, -1)  # 计算并保存当前步骤的中间状态。
        i, f, o, g = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o), torch.tanh(g)  # 计算并保存当前步骤的中间状态。
        c_new = f * c + i * g  # 计算并保存当前步骤的中间状态。
        h_new = o * torch.tanh(c_new)  # 计算并保存当前步骤的中间状态。
        return h_new, c_new  # 返回当前分支计算出的结果。

class ScratchBiLSTM(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, input_size, hidden_size):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.hidden_size = hidden_size  # 计算并保存当前步骤的中间状态。
        self.forward_cell = ScratchLSTMCell(input_size, hidden_size)  # 计算并保存当前步骤的中间状态。
        self.backward_cell = ScratchLSTMCell(input_size, hidden_size)  # 计算并保存当前步骤的中间状态。

    def _scan(self, x, mask, cell, reverse=False):  # 定义本节可复用的核心函数。
        B, T, _ = x.shape  # 计算并保存当前步骤的中间状态。
        h = x.new_zeros(B, self.hidden_size)  # 计算并保存当前步骤的中间状态。
        c = x.new_zeros(B, self.hidden_size)  # 计算并保存当前步骤的中间状态。
        outputs = [None] * T  # 计算并保存当前步骤的中间状态。
        time_ids = range(T - 1, -1, -1) if reverse else range(T)  # 计算并保存当前步骤的中间状态。
        for t in time_ids:  # 遍历输入元素以累积或检查结果。
            active = mask[:, t].unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
            h_new, c_new = cell(x[:, t], (h, c))  # 计算并保存当前步骤的中间状态。
            h, c = torch.where(active, h_new, h), torch.where(active, c_new, c)  # 计算并保存当前步骤的中间状态。
            outputs[t] = torch.where(active, h, torch.zeros_like(h))  # 计算并保存当前步骤的中间状态。
        return torch.stack(outputs, 1)  # 返回当前分支计算出的结果。

    def forward(self, x, mask):  # 定义本节可复用的核心函数。
        if x.ndim != 3 or mask.shape != x.shape[:2] or mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("期望 x=[B,T,D] 与 bool mask=[B,T]")  # 遇到非法合同立即显式失败。
        fw = self._scan(x, mask, self.forward_cell, reverse=False)  # 计算并保存当前步骤的中间状态。
        bw = self._scan(x, mask, self.backward_cell, reverse=True)  # 计算并保存当前步骤的中间状态。
        return torch.cat([fw, bw], -1)  # 返回当前分支计算出的结果。

bi0 = ScratchBiLSTM(5, 7)  # 计算并保存当前步骤的中间状态。
mask0 = torch.tensor([[1, 1, 1, 0], [1, 1, 0, 0]], dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
bi_out0 = bi0(torch.randn(2, 4, 5), mask0)  # 计算并保存当前步骤的中间状态。
assert bi_out0.shape == (2, 4, 14)  # 用受控断言验证关键不变量。
assert torch.count_nonzero(bi_out0[~mask0]) == 0  # 用受控断言验证关键不变量。
assert not any(isinstance(m, (nn.LSTM, nn.GRU, nn.RNN)) for m in bi0.modules())  # 用受控断言验证关键不变量。

## 3. 线性链 CRF 得分

路径 $y_{1:L}$ 的得分：

$$s(x,y)=a_{y_1}+\sum_{t=1}^{L}e_{t,y_t}+\sum_{t=2}^{L}A_{y_t,y_{t-1}}+b_{y_L}.$$

$a,b$ 是 START/END 分数，$A[next,prev]$ 是转移分数。条件概率为 $p(y|x)=\exp s(x,y)/Z(x)$，负对数似然是 $\log Z-s(x,y)$。所有动态规划都只更新 mask 为真的时间步。

## 4. log-partition：log-sum-exp 动态规划

初始化 $\alpha_1(k)=a_k+e_{1,k}$；递推：

$$\alpha_t(k)=e_{t,k}+\log\sum_j\exp(\alpha_{t-1}(j)+A_{k,j}).$$

最后 $\log Z=\log\sum_k\exp(\alpha_L(k)+b_k)$。约束通过把非法项替换为一个足够小的有限数；这里保留有限值以避免混合精度中 `inf-inf`。

In [ ]:
class LinearChainCRF(nn.Module):  # 定义承载本节状态与行为的数据结构。
    NEG = -1e4  # 计算并保存当前步骤的中间状态。
    def __init__(self, num_tags, allowed_transitions=None, allowed_start=None, allowed_end=None):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if num_tags <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("num_tags 必须为正")  # 遇到非法合同立即显式失败。
        self.num_tags = num_tags  # 计算并保存当前步骤的中间状态。
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))  # [next, prev]；中文说明：该行遵循既定约束。
        self.start = nn.Parameter(torch.empty(num_tags))  # 计算并保存当前步骤的中间状态。
        self.end = nn.Parameter(torch.empty(num_tags))  # 计算并保存当前步骤的中间状态。
        nn.init.uniform_(self.transitions, -0.1, 0.1)  # 执行当前语句以推进本节示例。
        nn.init.uniform_(self.start, -0.1, 0.1)  # 执行当前语句以推进本节示例。
        nn.init.uniform_(self.end, -0.1, 0.1)  # 执行当前语句以推进本节示例。
        at = torch.ones(num_tags, num_tags, dtype=torch.bool) if allowed_transitions is None else allowed_transitions.bool()  # 计算并保存当前步骤的中间状态。
        ast = torch.ones(num_tags, dtype=torch.bool) if allowed_start is None else allowed_start.bool()  # 计算并保存当前步骤的中间状态。
        aend = torch.ones(num_tags, dtype=torch.bool) if allowed_end is None else allowed_end.bool()  # 计算并保存当前步骤的中间状态。
        if at.shape != (num_tags, num_tags) or ast.shape != (num_tags,) or aend.shape != (num_tags,):  # 按当前条件选择后续控制路径。
            raise ValueError("CRF 约束矩阵/向量 shape 与 num_tags 不一致")  # 遇到非法合同立即显式失败。
        self.register_buffer("allowed_transitions", at)  # 执行当前语句以推进本节示例。
        self.register_buffer("allowed_start", ast)  # 执行当前语句以推进本节示例。
        self.register_buffer("allowed_end", aend)  # 执行当前语句以推进本节示例。

    def _validate(self, emissions, mask, tags=None):  # 定义本节可复用的核心函数。
        if emissions.ndim != 3 or emissions.shape[-1] != self.num_tags:  # 按当前条件选择后续控制路径。
            raise ValueError("emissions 必须为 [B,T,K]")  # 遇到非法合同立即显式失败。
        if mask.shape != emissions.shape[:2] or mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("mask 必须是 [B,T] bool")  # 遇到非法合同立即显式失败。
        if bool((~mask[:, 0]).any()):  # 按当前条件选择后续控制路径。
            raise ValueError("CRF 拒绝空序列")  # 遇到非法合同立即显式失败。
        if bool(((~mask[:, :-1]) & mask[:, 1:]).any()):  # 按当前条件选择后续控制路径。
            raise ValueError("mask 必须是连续左前缀")  # 遇到非法合同立即显式失败。
        if tags is not None:  # 按当前条件选择后续控制路径。
            if tags.shape != mask.shape:  # 按当前条件选择后续控制路径。
                raise ValueError("tags shape 不匹配")  # 遇到非法合同立即显式失败。
            if tags.dtype != torch.long:  # 按当前条件选择后续控制路径。
                raise ValueError("tags 必须为 torch.long")  # 遇到非法合同立即显式失败。
            if bool((((tags < 0) | (tags >= self.num_tags)) & mask).any()):  # 按当前条件选择后续控制路径。
                raise ValueError("有效位置的 tag id 越界")  # 遇到非法合同立即显式失败。

    def constrained(self):  # 定义本节可复用的核心函数。
        trans = self.transitions.masked_fill(~self.allowed_transitions, self.NEG)  # 计算并保存当前步骤的中间状态。
        start = self.start.masked_fill(~self.allowed_start, self.NEG)  # 计算并保存当前步骤的中间状态。
        end = self.end.masked_fill(~self.allowed_end, self.NEG)  # 计算并保存当前步骤的中间状态。
        return trans, start, end  # 返回当前分支计算出的结果。

    def ensure_legal_path(self, mask):  # 定义本节可复用的核心函数。
        # 仅在布尔约束图上做可达性 DP，不让有限 NEG 伪装成一条低分合法路径。
        reachable = self.allowed_start.unsqueeze(0).expand(mask.shape[0], -1)  # 计算并保存当前步骤的中间状态。
        for t in range(1, mask.shape[1]):  # 遍历输入元素以累积或检查结果。
            next_reachable = (self.allowed_transitions.unsqueeze(0)  # 计算并保存当前步骤的中间状态。
                              & reachable[:, None, :]).any(dim=-1)  # 计算并保存当前步骤的中间状态。
            reachable = torch.where(mask[:, t, None], next_reachable, reachable)  # 计算并保存当前步骤的中间状态。
        has_complete_path = (reachable & self.allowed_end.unsqueeze(0)).any(dim=-1)  # 计算并保存当前步骤的中间状态。
        if bool((~has_complete_path).any()):  # 按当前条件选择后续控制路径。
            bad_rows = (~has_complete_path).nonzero(as_tuple=False).flatten().tolist()  # 计算并保存当前步骤的中间状态。
            raise ValueError(f"约束图中不存在完整合法路径，样本索引: {bad_rows}")  # 遇到非法合同立即显式失败。
        return reachable  # 返回当前分支计算出的结果。

    def log_partition(self, emissions, mask):  # 定义本节可复用的核心函数。
        self._validate(emissions, mask)  # 执行当前语句以推进本节示例。
        self.ensure_legal_path(mask)  # 执行当前语句以推进本节示例。
        trans, start, end = self.constrained()  # 计算并保存当前步骤的中间状态。
        alpha = start + emissions[:, 0]  # 计算并保存当前步骤的中间状态。
        for t in range(1, emissions.shape[1]):  # 遍历输入元素以累积或检查结果。
            scores = alpha[:, None, :] + trans[None, :, :] + emissions[:, t, :, None]  # 计算并保存当前步骤的中间状态。
            next_alpha = torch.logsumexp(scores, dim=-1)  # 计算并保存当前步骤的中间状态。
            alpha = torch.where(mask[:, t, None], next_alpha, alpha)  # 计算并保存当前步骤的中间状态。
        return torch.logsumexp(alpha + end, dim=-1)  # 返回当前分支计算出的结果。

    def gold_score(self, emissions, tags, mask):  # 定义本节可复用的核心函数。
        self._validate(emissions, mask, tags)  # 执行当前语句以推进本节示例。
        trans, start, end = self.constrained()  # 计算并保存当前步骤的中间状态。
        first = tags[:, 0]  # 计算并保存当前步骤的中间状态。
        if bool((~self.allowed_start[first]).any()):  # 按当前条件选择后续控制路径。
            raise ValueError("gold 路径含非法 START 转移")  # 遇到非法合同立即显式失败。
        score = start[first] + emissions[:, 0].gather(1, first[:, None]).squeeze(1)  # 计算并保存当前步骤的中间状态。
        prev = first  # 计算并保存当前步骤的中间状态。
        for t in range(1, emissions.shape[1]):  # 遍历输入元素以累积或检查结果。
            cur, active = tags[:, t], mask[:, t]  # 计算并保存当前步骤的中间状态。
            cur_safe = torch.where(active, cur, prev)  # padding tag 的具体填充值不应参与索引
            legal = self.allowed_transitions[cur_safe, prev]  # 计算并保存当前步骤的中间状态。
            if bool((active & ~legal).any()):  # 按当前条件选择后续控制路径。
                raise ValueError("gold 路径含非法标签转移")  # 遇到非法合同立即显式失败。
            step = trans[cur_safe, prev] + emissions[:, t].gather(1, cur_safe[:, None]).squeeze(1)  # 计算并保存当前步骤的中间状态。
            score = score + torch.where(active, step, torch.zeros_like(step))  # 计算并保存当前步骤的中间状态。
            prev = cur_safe  # 计算并保存当前步骤的中间状态。
        if bool((~self.allowed_end[prev]).any()):  # 按当前条件选择后续控制路径。
            raise ValueError("gold 路径含非法 END 转移")  # 遇到非法合同立即显式失败。
        return score + end[prev]  # 返回当前分支计算出的结果。

    def neg_log_likelihood(self, emissions, tags, mask):  # 定义本节可复用的核心函数。
        return (self.log_partition(emissions, mask) - self.gold_score(emissions, tags, mask)).mean()  # 返回当前分支计算出的结果。

assert LinearChainCRF(3).transitions.shape == (3, 3)  # 用受控断言验证关键不变量。
assert LinearChainCRF.NEG < -1000  # 用受控断言验证关键不变量。

## 5. Viterbi：把求和换成最大值

Viterbi 与 log-partition 共享状态图，但递推取 `max` 并保存每步 backpointer。变长 batch 在样本结束后冻结分数；回溯只使用各自有效长度。返回 Python 标签列表，便于后处理成实体 span。

In [ ]:
def crf_viterbi(self, emissions, mask):  # 定义本节可复用的核心函数。
    self._validate(emissions, mask)  # 执行当前语句以推进本节示例。
    self.ensure_legal_path(mask)  # 执行当前语句以推进本节示例。
    trans, start, end = self.constrained()  # 计算并保存当前步骤的中间状态。
    score = start + emissions[:, 0]  # 计算并保存当前步骤的中间状态。
    backpointers = []  # 计算并保存当前步骤的中间状态。
    for t in range(1, emissions.shape[1]):  # 遍历输入元素以累积或检查结果。
        candidates = score[:, None, :] + trans[None, :, :]  # 计算并保存当前步骤的中间状态。
        best_score, best_prev = candidates.max(dim=-1)  # 计算并保存当前步骤的中间状态。
        best_score = best_score + emissions[:, t]  # 计算并保存当前步骤的中间状态。
        score = torch.where(mask[:, t, None], best_score, score)  # 计算并保存当前步骤的中间状态。
        backpointers.append(best_prev)  # 执行当前语句以推进本节示例。
    score = score + end  # 计算并保存当前步骤的中间状态。
    best_last = score.argmax(-1)  # 计算并保存当前步骤的中间状态。
    lengths = mask.sum(1).tolist()  # 计算并保存当前步骤的中间状态。
    paths = []  # 计算并保存当前步骤的中间状态。
    for b, length in enumerate(lengths):  # 遍历输入元素以累积或检查结果。
        tag = int(best_last[b])  # 计算并保存当前步骤的中间状态。
        path = [tag]  # 计算并保存当前步骤的中间状态。
        for t in range(length - 1, 0, -1):  # 遍历输入元素以累积或检查结果。
            tag = int(backpointers[t - 1][b, tag])  # 计算并保存当前步骤的中间状态。
            path.append(tag)  # 执行当前语句以推进本节示例。
        paths.append(list(reversed(path)))  # 执行当前语句以推进本节示例。
    return paths, score.max(-1).values  # 返回当前分支计算出的结果。

LinearChainCRF.viterbi_decode = crf_viterbi  # 计算并保存当前步骤的中间状态。
crf0 = LinearChainCRF(3)  # 计算并保存当前步骤的中间状态。
em0 = torch.randn(2, 4, 3)  # 计算并保存当前步骤的中间状态。
paths0, scores0 = crf0.viterbi_decode(em0, mask0)  # 计算并保存当前步骤的中间状态。
assert list(map(len, paths0)) == [3, 2]  # 用受控断言验证关键不变量。
assert scores0.shape == (2,)  # 用受控断言验证关键不变量。
assert all(0 <= tag < 3 for path in paths0 for tag in path)  # 用受控断言验证关键不变量。

## 6. 穷举对照：给动态规划一个独立 oracle

长度 3、标签数 3 只有 $3^3=27$ 条路径，可以逐条算分。我们把穷举的 `logsumexp` 与最大路径分别对照 CRF 的 `log_partition` 和 Viterbi。这个测试能同时发现转移矩阵方向、START/END 和 emission 索引错误。

In [ ]:
def brute_force_scores(crf, emissions_1d):  # 定义本节可复用的核心函数。
    trans, start, end = crf.constrained()  # 计算并保存当前步骤的中间状态。
    scored = []  # 计算并保存当前步骤的中间状态。
    for path in itertools.product(range(crf.num_tags), repeat=emissions_1d.shape[0]):  # 遍历输入元素以累积或检查结果。
        if not crf.allowed_start[path[0]] or not crf.allowed_end[path[-1]]:  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        if any(not crf.allowed_transitions[path[t], path[t-1]] for t in range(1, len(path))):  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        score = start[path[0]] + emissions_1d[0, path[0]]  # 计算并保存当前步骤的中间状态。
        for t in range(1, len(path)):  # 遍历输入元素以累积或检查结果。
            score = score + trans[path[t], path[t-1]] + emissions_1d[t, path[t]]  # 计算并保存当前步骤的中间状态。
        score = score + end[path[-1]]  # 计算并保存当前步骤的中间状态。
        scored.append((path, score))  # 执行当前语句以推进本节示例。
    return scored  # 返回当前分支计算出的结果。

torch.manual_seed(SEED + 1)  # 执行当前语句以推进本节示例。
tiny_crf = LinearChainCRF(3)  # 计算并保存当前步骤的中间状态。
tiny_em = torch.randn(1, 3, 3)  # 计算并保存当前步骤的中间状态。
tiny_mask = torch.ones(1, 3, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
enumerated = brute_force_scores(tiny_crf, tiny_em[0])  # 计算并保存当前步骤的中间状态。
brute_logz = torch.logsumexp(torch.stack([s for _, s in enumerated]), 0)  # 计算并保存当前步骤的中间状态。
brute_path, brute_best = max(enumerated, key=lambda pair: float(pair[1]))  # 计算并保存当前步骤的中间状态。
dp_logz = tiny_crf.log_partition(tiny_em, tiny_mask)[0]  # 计算并保存当前步骤的中间状态。
dp_paths, dp_scores = tiny_crf.viterbi_decode(tiny_em, tiny_mask)  # 计算并保存当前步骤的中间状态。

assert len(enumerated) == 27  # 用受控断言验证关键不变量。
assert torch.allclose(dp_logz, brute_logz, atol=1e-6)  # 用受控断言验证关键不变量。
assert tuple(dp_paths[0]) == brute_path  # 用受控断言验证关键不变量。
assert torch.allclose(dp_scores[0], brute_best, atol=1e-6)  # 用受控断言验证关键不变量。
assert dp_logz >= dp_scores[0]  # 用受控断言验证关键不变量。

## 7. BIO 合法转移

对单一实体类型，START 不能直接进入 `I-X`，`O -> I-X` 也非法；其他转移在这个简化协议中允许。复杂 NER 还要禁止 `B-PER -> I-ORG` 等跨类型连接。约束既用于分母也用于解码，并对 gold 路径做显式拒绝。

In [ ]:
allowed_trans = torch.ones(3, 3, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
allowed_trans[I_X, O] = False       # transition[next=I, prev=O]；中文说明：该行遵循既定约束。
allowed_start = torch.tensor([True, True, False])  # 计算并保存当前步骤的中间状态。
allowed_end = torch.tensor([True, True, True])  # 计算并保存当前步骤的中间状态。
bio_crf = LinearChainCRF(3, allowed_trans, allowed_start, allowed_end)  # 计算并保存当前步骤的中间状态。

illegal_tags = torch.tensor([[I_X, O]])  # 计算并保存当前步骤的中间状态。
illegal_em = torch.randn(1, 2, 3)  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    bio_crf.gold_score(illegal_em, illegal_tags, torch.ones(1, 2, dtype=torch.bool))  # 计算并保存当前步骤的中间状态。
    illegal_start_rejected = False  # 计算并保存当前步骤的中间状态。
except ValueError:  # 捕获预期异常并验证失败分支。
    illegal_start_rejected = True  # 计算并保存当前步骤的中间状态。

assert illegal_start_rejected  # 用受控断言验证关键不变量。
assert not bio_crf.allowed_start[I_X]  # 用受控断言验证关键不变量。
assert not bio_crf.allowed_transitions[I_X, O]  # 用受控断言验证关键不变量。
assert bio_crf.allowed_transitions[I_X, B_X]  # 用受控断言验证关键不变量。

## 8. BiLSTM-CRF 组合

Embedding 把 token 映射到 `[B,T,E]`，手写 BiLSTM 生成 `[B,T,2H]`，线性层产生 emission。训练调用 CRF NLL；推理调用 Viterbi。padding token 的 embedding 固定为零，但真正的边界仍由 mask 决定。

若词表 $V$、嵌入 $E$、单向隐藏维 $H$、标签数 $K$，参数量为：embedding $VE$；双向 LSTM $2	imes4H(E+H+1)$；emission $K(2H+1)$；CRF $K^2+2K$。

In [ ]:
class BiLSTMCRF(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab_size, embed_dim, hidden_size, num_tags,  # 定义本节可复用的核心函数。
                 allowed_transitions, allowed_start, allowed_end):  # 执行当前语句以推进本节示例。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.config = dict(vocab_size=vocab_size, embed_dim=embed_dim,  # 计算并保存当前步骤的中间状态。
                           hidden_size=hidden_size, num_tags=num_tags)  # 计算并保存当前步骤的中间状态。
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)  # 计算并保存当前步骤的中间状态。
        self.encoder = ScratchBiLSTM(embed_dim, hidden_size)  # 计算并保存当前步骤的中间状态。
        self.emission = nn.Linear(2 * hidden_size, num_tags)  # 计算并保存当前步骤的中间状态。
        self.crf = LinearChainCRF(num_tags, allowed_transitions, allowed_start, allowed_end)  # 计算并保存当前步骤的中间状态。

    def emissions(self, tokens, mask):  # 定义本节可复用的核心函数。
        if tokens.shape != mask.shape:  # 按当前条件选择后续控制路径。
            raise ValueError("tokens 与 mask shape 不一致")  # 遇到非法合同立即显式失败。
        return self.emission(self.encoder(self.embedding(tokens), mask))  # 返回当前分支计算出的结果。

    def forward(self, tokens, mask, tags=None):  # 定义本节可复用的核心函数。
        emissions = self.emissions(tokens, mask)  # 计算并保存当前步骤的中间状态。
        if tags is None:  # 按当前条件选择后续控制路径。
            return self.crf.viterbi_decode(emissions, mask)[0]  # 返回当前分支计算出的结果。
        return self.crf.neg_log_likelihood(emissions, tags, mask)  # 返回当前分支计算出的结果。

model_probe = BiLSTMCRF(12, 8, 10, 3, allowed_trans, allowed_start, allowed_end)  # 计算并保存当前步骤的中间状态。
probe_em = model_probe.emissions(torch.tensor([[2, 3, 0]]), torch.tensor([[1, 1, 0]], dtype=torch.bool))  # 计算并保存当前步骤的中间状态。
parameter_breakdown21 = {  # 计算并保存当前步骤的中间状态。
    "embedding": sum(p.numel() for p in model_probe.embedding.parameters()),  # 执行当前语句以推进本节示例。
    "bilstm": sum(p.numel() for p in model_probe.encoder.parameters()),  # 执行当前语句以推进本节示例。
    "emission": sum(p.numel() for p in model_probe.emission.parameters()),  # 执行当前语句以推进本节示例。
    "crf": sum(p.numel() for p in model_probe.crf.parameters()),  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
assert probe_em.shape == (1, 3, 3)  # 用受控断言验证关键不变量。
assert not any(isinstance(m, (nn.LSTM, nn.GRU, nn.RNN)) for m in model_probe.modules())  # 用受控断言验证关键不变量。
assert parameter_breakdown21["crf"] == 3 * 3 + 2 * 3  # 用受控断言验证关键不变量。
assert sum(parameter_breakdown21.values()) == sum(p.numel() for p in model_probe.parameters())  # 用受控断言验证关键不变量。
print(parameter_breakdown21)  # 执行当前语句以推进本节示例。

## 9. 小型 NER 受控过拟合集

token 语义：`1` 是 padding；实际 padding 仍使用 `0`。`2/3/4` 是普通词，`5/6` 是实体首词，`7/8` 是实体续词。每条序列左对齐，标签遵守 BIO。集合刻意覆盖单 token 实体、多 token 实体、句首/句中实体和不同长度。

这不是独立测试集。它仅用于证明 emission、CRF loss 与 Viterbi 能协同记住一个合法的小集合。

In [ ]:
token_rows = [  # 计算并保存当前步骤的中间状态。
    [2, 5, 7, 3], [5, 7, 8, 4], [2, 3, 5], [6, 8, 4, 2, 3],  # 执行当前语句以推进本节示例。
    [4, 2], [3, 6, 7], [5, 2, 3, 4], [2, 6, 8, 3, 4],  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
tag_rows = [  # 计算并保存当前步骤的中间状态。
    [O, B_X, I_X, O], [B_X, I_X, I_X, O], [O, O, B_X], [B_X, I_X, O, O, O],  # 执行当前语句以推进本节示例。
    [O, O], [O, B_X, I_X], [B_X, O, O, O], [O, B_X, I_X, O, O],  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
max_t = max(map(len, token_rows))  # 计算并保存当前步骤的中间状态。
tokens21 = torch.zeros(len(token_rows), max_t, dtype=torch.long)  # 计算并保存当前步骤的中间状态。
tags21 = torch.zeros_like(tokens21)  # 计算并保存当前步骤的中间状态。
mask21 = torch.zeros_like(tokens21, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
for i, (tokens, tags) in enumerate(zip(token_rows, tag_rows)):  # 遍历输入元素以累积或检查结果。
    tokens21[i, :len(tokens)] = torch.tensor(tokens)  # 计算并保存当前步骤的中间状态。
    tags21[i, :len(tags)] = torch.tensor(tags)  # 计算并保存当前步骤的中间状态。
    mask21[i, :len(tokens)] = True  # 计算并保存当前步骤的中间状态。

assert tokens21.shape == tags21.shape == mask21.shape == (8, 5)  # 用受控断言验证关键不变量。
assert mask21.sum(1).tolist() == list(map(len, token_rows))  # 用受控断言验证关键不变量。
assert not ((tags21 == I_X) & ~mask21).any()  # 用受控断言验证关键不变量。
assert all(tags[0] != I_X for tags in tag_rows)  # 用受控断言验证关键不变量。

## 10. NLL 优化、梯度与裁剪

Adam 最小化 batch 平均 CRF NLL。动态规划全程可微，梯度应流向 embedding、两个方向的 LSTM、emission 和转移参数。首步检查非零有限梯度，随后裁剪全局范数。

In [ ]:
torch.manual_seed(SEED + 2)  # 执行当前语句以推进本节示例。
model21 = BiLSTMCRF(12, 12, 14, 3, allowed_trans, allowed_start, allowed_end)  # 计算并保存当前步骤的中间状态。
opt21 = torch.optim.Adam(model21.parameters(), lr=0.025)  # 计算并保存当前步骤的中间状态。
losses21 = []  # 计算并保存当前步骤的中间状态。
model21.train()  # 执行当前语句以推进本节示例。
for step in range(220):  # 遍历输入元素以累积或检查结果。
    opt21.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    loss = model21(tokens21, mask21, tags21)  # 计算并保存当前步骤的中间状态。
    loss.backward()  # 执行当前语句以推进本节示例。
    if step == 0:  # 按当前条件选择后续控制路径。
        named_grads = {n: p.grad for n, p in model21.named_parameters() if p.grad is not None}  # 计算并保存当前步骤的中间状态。
        assert named_grads  # 用受控断言验证关键不变量。
        assert all(torch.isfinite(g).all() for g in named_grads.values())  # 用受控断言验证关键不变量。
        assert sum(float(g.abs().sum()) for g in named_grads.values()) > 0  # 用受控断言验证关键不变量。
        assert model21.crf.transitions.grad.abs().sum() > 0  # 用受控断言验证关键不变量。
    torch.nn.utils.clip_grad_norm_(model21.parameters(), 1.0)  # 执行当前语句以推进本节示例。
    opt21.step()  # 执行当前语句以推进本节示例。
    losses21.append(float(loss.detach()))  # 执行当前语句以推进本节示例。

model21.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    pred_paths21 = model21(tokens21, mask21)  # 计算并保存当前步骤的中间状态。

gold_paths21 = [tags[:len(tokens)] for tags, tokens in zip(tag_rows, token_rows)]  # 计算并保存当前步骤的中间状态。
token_correct = sum(p == g for pred, gold in zip(pred_paths21, gold_paths21) for p, g in zip(pred, gold))  # 计算并保存当前步骤的中间状态。
token_total = sum(map(len, gold_paths21))  # 计算并保存当前步骤的中间状态。
token_acc21 = token_correct / token_total  # 计算并保存当前步骤的中间状态。

assert losses21[-1] < losses21[0] * 0.1  # 用受控断言验证关键不变量。
assert token_acc21 >= 0.97  # 用受控断言验证关键不变量。
assert list(map(len, pred_paths21)) == list(map(len, gold_paths21))  # 用受控断言验证关键不变量。
print({"loss_first": losses21[0], "loss_last": losses21[-1], "controlled_token_acc": token_acc21})  # 执行当前语句以推进本节示例。

## 11. 从 BIO 标签恢复 span，并计算实体级 F1

token accuracy 会被大量 `O` 稀释，所以 NER 主要看实体 span 的精确匹配。简化规则：`B-X` 开启实体，连续 `I-X` 延长；孤立 `I-X` 视为非法并拒绝。计算 micro precision/recall/F1，而不是把每句 F1 简单平均。

In [ ]:
def bio_to_spans(tags):  # 定义本节可复用的核心函数。
    spans, start = [], None  # 计算并保存当前步骤的中间状态。
    for i, tag in enumerate(list(tags) + [O]):  # 遍历输入元素以累积或检查结果。
        if tag == B_X:  # 按当前条件选择后续控制路径。
            if start is not None:  # 按当前条件选择后续控制路径。
                spans.append((start, i, "X"))  # 执行当前语句以推进本节示例。
            start = i  # 计算并保存当前步骤的中间状态。
        elif tag == I_X:  # 按当前条件选择后续控制路径。
            if start is None:  # 按当前条件选择后续控制路径。
                raise ValueError("孤立 I-X 不是合法 BIO")  # 遇到非法合同立即显式失败。
        else:  # 处理前置条件不成立的分支。
            if start is not None:  # 按当前条件选择后续控制路径。
                spans.append((start, i, "X"))  # 执行当前语句以推进本节示例。
                start = None  # 计算并保存当前步骤的中间状态。
    return spans  # 返回当前分支计算出的结果。

gold_spans = {(i, *span) for i, tags in enumerate(gold_paths21) for span in bio_to_spans(tags)}  # 计算并保存当前步骤的中间状态。
pred_spans = {(i, *span) for i, tags in enumerate(pred_paths21) for span in bio_to_spans(tags)}  # 计算并保存当前步骤的中间状态。
tp = len(gold_spans & pred_spans)  # 计算并保存当前步骤的中间状态。
precision21 = tp / len(pred_spans) if pred_spans else 0.0  # 计算并保存当前步骤的中间状态。
recall21 = tp / len(gold_spans) if gold_spans else 0.0  # 计算并保存当前步骤的中间状态。
span_f1_21 = 2 * precision21 * recall21 / (precision21 + recall21) if precision21 + recall21 else 0.0  # 计算并保存当前步骤的中间状态。

assert gold_spans  # 用受控断言验证关键不变量。
assert 0.0 <= precision21 <= 1.0  # 用受控断言验证关键不变量。
assert 0.0 <= recall21 <= 1.0  # 用受控断言验证关键不变量。
assert span_f1_21 >= 0.95  # 用受控断言验证关键不变量。
assert bio_to_spans([O, B_X, I_X, O]) == [(1, 3, "X")]  # 用受控断言验证关键不变量。
print({"span_precision": precision21, "span_recall": recall21, "span_f1": span_f1_21})  # 执行当前语句以推进本节示例。

## 12. 失败反例与数值边界

- 把转移写成 `[prev,next]` 却按 `[next,prev]` 索引：loss 仍可能下降，但 Viterbi 路径错误。
- 分母允许非法路径、分子拒绝非法路径：概率空间不一致。
- 约束图对某个序列长度根本没有 START→…→END 路径，却仍用有限大负数继续解码：会把非法低分路径伪装成结果；必须先做布尔可达性检查并 fail closed。
- padding 后又出现有效 token：冻结动态规划会静默丢失中间洞，因此必须拒绝非前缀 mask。
- 用逐 token softmax 替代 CRF：无法建模标签转移约束。
- 在 fp16 直接使用真正的负无穷并做不稳定运算：容易出现 NaN；生产实现应验证 dtype 策略。
- 仅看 token accuracy：全预测 O 也可能很高，必须报告 span F1。

In [ ]:
def rejected(fn):  # 定义本节可复用的核心函数。
    try:  # 尝试执行可能失败的受控操作。
        fn()  # 执行当前语句以推进本节示例。
        return False  # 返回当前分支计算出的结果。
    except ValueError:  # 捕获预期异常并验证失败分支。
        return True  # 返回当前分支计算出的结果。

empty_mask = torch.tensor([[False, False]])  # 计算并保存当前步骤的中间状态。
hole_mask = torch.tensor([[True, False, True]])  # 计算并保存当前步骤的中间状态。
no_start_crf = LinearChainCRF(2, allowed_start=torch.zeros(2, dtype=torch.bool))  # 计算并保存当前步骤的中间状态。
dead_end_crf = LinearChainCRF(  # 计算并保存当前步骤的中间状态。
    2, allowed_transitions=torch.zeros(2, 2, dtype=torch.bool),  # 计算并保存当前步骤的中间状态。
    allowed_start=torch.tensor([True, False]), allowed_end=torch.ones(2, dtype=torch.bool),  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
assert rejected(lambda: model21.crf.log_partition(torch.randn(1, 2, 3), empty_mask))  # 用受控断言验证关键不变量。
assert rejected(lambda: model21.crf.log_partition(torch.randn(1, 3, 3), hole_mask))  # 用受控断言验证关键不变量。
assert rejected(lambda: no_start_crf.log_partition(  # 用受控断言验证关键不变量。
    torch.randn(1, 1, 2), torch.ones(1, 1, dtype=torch.bool)))  # 计算并保存当前步骤的中间状态。
assert rejected(lambda: no_start_crf.viterbi_decode(  # 用受控断言验证关键不变量。
    torch.randn(1, 1, 2), torch.ones(1, 1, dtype=torch.bool)))  # 计算并保存当前步骤的中间状态。
assert torch.isfinite(dead_end_crf.log_partition(  # 用受控断言验证关键不变量。
    torch.randn(1, 1, 2), torch.ones(1, 1, dtype=torch.bool))).all()  # 计算并保存当前步骤的中间状态。
assert rejected(lambda: dead_end_crf.log_partition(  # 用受控断言验证关键不变量。
    torch.randn(1, 2, 2), torch.ones(1, 2, dtype=torch.bool)))  # 计算并保存当前步骤的中间状态。
assert rejected(lambda: dead_end_crf.viterbi_decode(  # 用受控断言验证关键不变量。
    torch.randn(1, 2, 2), torch.ones(1, 2, dtype=torch.bool)))  # 计算并保存当前步骤的中间状态。
assert rejected(lambda: bio_to_spans([I_X, O]))  # 用受控断言验证关键不变量。
assert rejected(lambda: LinearChainCRF(3, torch.ones(2, 2, dtype=torch.bool)))  # 用受控断言验证关键不变量。
assert rejected(lambda: model21.crf.gold_score(torch.randn(1, 2, 3),  # 用受控断言验证关键不变量。
                                                torch.zeros(1, 2),  # 执行当前语句以推进本节示例。
                                                torch.ones(1, 2, dtype=torch.bool)))  # 计算并保存当前步骤的中间状态。
assert rejected(lambda: model21.emissions(torch.ones(1, 2, dtype=torch.long),  # 用受控断言验证关键不变量。
                                          torch.ones(1, 3, dtype=torch.bool)))  # 计算并保存当前步骤的中间状态。
assert torch.isfinite(model21.crf.transitions).all()  # 用受控断言验证关键不变量。

## 13. 制品合同与指纹

除了网络维度，必须固化 tag-to-id、BIO 约束矩阵、转移方向、padding id、mask 语义和解码算法版本。任何一项变化都会让相同权重产生不同实体边界。下面在内存保存并重新加载，避免依赖外部文件。

In [ ]:
manifest21 = {  # 计算并保存当前步骤的中间状态。
    "artifact": "bilstm_crf_from_scratch",  # 执行当前语句以推进本节示例。
    "schema_version": 1,  # 执行当前语句以推进本节示例。
    "config": model21.config,  # 执行当前语句以推进本节示例。
    "tags": TAG_NAMES,  # 执行当前语句以推进本节示例。
    "padding_id": 0,  # 执行当前语句以推进本节示例。
    "lstm_gate_order": ScratchLSTMCell.gate_order,  # 执行当前语句以推进本节示例。
    "transition_layout": "transition[next_tag, previous_tag]",  # 执行当前语句以推进本节示例。
    "mask_contract": "non-empty contiguous left prefix",  # 执行当前语句以推进本节示例。
    "torch_version": torch.__version__,  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
buf21 = io.BytesIO()  # 计算并保存当前步骤的中间状态。
torch.save({"manifest": manifest21, "state_dict": model21.state_dict()}, buf21)  # 执行当前语句以推进本节示例。
artifact21 = buf21.getvalue()  # 计算并保存当前步骤的中间状态。
sha21 = hashlib.sha256(artifact21).hexdigest()  # 计算并保存当前步骤的中间状态。
buf21.seek(0)  # 执行当前语句以推进本节示例。
loaded21 = torch.load(buf21, map_location="cpu", weights_only=False)  # 计算并保存当前步骤的中间状态。
clone21 = BiLSTMCRF(**loaded21["manifest"]["config"],  # 计算并保存当前步骤的中间状态。
                    allowed_transitions=allowed_trans,  # 计算并保存当前步骤的中间状态。
                    allowed_start=allowed_start,  # 计算并保存当前步骤的中间状态。
                    allowed_end=allowed_end)  # 计算并保存当前步骤的中间状态。
clone21.load_state_dict(loaded21["state_dict"])  # 执行当前语句以推进本节示例。
clone21.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    clone_paths21 = clone21(tokens21, mask21)  # 计算并保存当前步骤的中间状态。

assert len(sha21) == 64  # 用受控断言验证关键不变量。
assert clone_paths21 == pred_paths21  # 用受控断言验证关键不变量。
assert loaded21["manifest"]["transition_layout"].startswith("transition[next_tag")  # 用受控断言验证关键不变量。
assert set(clone21.state_dict()) == set(model21.state_dict())  # 用受控断言验证关键不变量。
print({"sha256": sha21[:16] + "…", "bytes": len(artifact21)})  # 执行当前语句以推进本节示例。

## 14. 生产替换、安全与观测

手写 Python 时间循环适合学习和小规模校验，不适合长序列吞吐。生产可换成官方融合 BiLSTM 与经过审计的 CRF，但必须保留穷举 oracle、变长 batch、非法路径和 span 指标回归。

服务入口限制最大序列长度、词表范围和 batch；不加载不可信 pickle；记录未知 token 率、长度/截断分布、非法 BIO 率、span 数量、各标签比例、NLL、梯度范数、延迟和制品哈希。涉及人名等敏感实体时，日志应去标识化并做租户隔离。

## 15. 原始论文与官方文档

- Hochreiter & Schmidhuber, *Long Short-Term Memory* (1997)：https://doi.org/10.1162/neco.1997.9.8.1735
- Lafferty et al., *Conditional Random Fields* (2001)：https://repository.upenn.edu/cis_papers/159/
- Huang et al., *Bidirectional LSTM-CRF Models for Sequence Tagging* (2015)：https://arxiv.org/abs/1508.01991
- PyTorch `nn.Module`：https://pytorch.org/docs/stable/generated/torch.nn.Module.html
- PyTorch `logsumexp`：https://pytorch.org/docs/stable/generated/torch.logsumexp.html

模型思路与概率公式来自上述论文；代码和测试为本笔记重新实现。

In [ ]:
# 最终合同检查：分数关系、参数梯度、约束与指纹。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    final_em21 = model21.emissions(tokens21, mask21)  # 计算并保存当前步骤的中间状态。
    final_logz21 = model21.crf.log_partition(final_em21, mask21)  # 计算并保存当前步骤的中间状态。
    final_gold21 = model21.crf.gold_score(final_em21, tags21, mask21)  # 计算并保存当前步骤的中间状态。
    padded_tags21 = tags21.clone()  # 计算并保存当前步骤的中间状态。
    padded_tags21[~mask21] = -100  # 计算并保存当前步骤的中间状态。
    padded_gold21 = model21.crf.gold_score(final_em21, padded_tags21, mask21)  # 计算并保存当前步骤的中间状态。

assert final_logz21.shape == final_gold21.shape == (8,)  # 用受控断言验证关键不变量。
assert torch.all(final_logz21 + 1e-5 >= final_gold21)  # 用受控断言验证关键不变量。
assert torch.allclose(final_gold21, padded_gold21)  # 用受控断言验证关键不变量。
assert losses21[-1] >= -1e-5  # 用受控断言验证关键不变量。
assert token_total == int(mask21.sum())  # 用受控断言验证关键不变量。
assert manifest21["tags"] == TAG_NAMES  # 用受控断言验证关键不变量。
assert manifest21["lstm_gate_order"] == "ifgo"  # 用受控断言验证关键不变量。
assert sha21 == hashlib.sha256(artifact21).hexdigest()  # 用受控断言验证关键不变量。
assert all(torch.isfinite(p).all() for p in model21.parameters())  # 用受控断言验证关键不变量。
assert pred_paths21 == clone_paths21  # 用受控断言验证关键不变量。
print("Notebook 21：全部合同测试通过。")  # 执行当前语句以推进本节示例。